# How MemBukkit works — the explainability tour

MemBukkit is built so you can always answer *"why did it say that?"*. This notebook walks the full pipeline on the bundled demo datasets:

1. **Two lanes** — every ingested session is stored twice: raw turns (*verbatim* lane) and LLM-distilled dated facts (*atomic* lane). Different questions need different evidence.
2. **Topic buckets** — each lane's facts are clustered into topic buckets. Retrieval *routes* to the most promising buckets instead of scanning everything.
3. **Scan budget** — a hard cap on the fraction of memory scanned per query. This is the efficiency lever: cost degrades gracefully as you tighten it.
4. **Traces** — every answer carries a `RetrievalTrace`: which buckets opened, what fraction of memory was scanned, and the exact ranked, dated facts the reader saw.
5. **Provenance** — every fact links back to its source document and the exact turn inside it.

In [ ]:
from membukkit import MemorySystem, RetrievalConfig
from membukkit.ingest import parse_path
from membukkit.storage import LocalStore

mem = MemorySystem.from_pretrained(llm="openai:gpt-4o-mini")
store = LocalStore("xai-tour")

# Ingest all four bundled demos into ONE memory bank so buckets are meaningful:
# personal chat + support tickets + contracts + engineering docs.
docs = parse_path("../demos")
docs = [d for d in docs if d.sessions]
for doc in docs:
    doc_id = store.add_document(doc.name, doc.sessions, doc.dates, doc_type=doc.doc_type)
    mem.ingest(doc.sessions, dates=doc.dates, doc_id=doc_id, doc_name=doc.name)
store.save_backend(mem.backend)

print(f"{len(docs)} documents -> {mem.backend.count()} facts "
      f"({mem.backend.count_kind('verbatim')} verbatim, {mem.backend.count_kind('atomic')} atomic)")

## The memory map: topic buckets

KMeans over fact embeddings partitions each lane into topic buckets. Ask the LLM to label them and you get a human-readable map of everything the system remembers.

In [ ]:
part = mem.partition()
labels = mem.label_buckets()

by_bucket = part.get("by_bucket", {})
for b in range(part.get("k_eff", 0)):
    exemplar = (mem.backend.topic_exemplars(b, n=1) or [""])[0]
    print(f"[{b:>2}] {len(by_bucket.get(b, [])):>4} facts  {labels.get(b, ''):<32} e.g. {exemplar[:70]}")

## Ask, then open the trace

The answer is only half the output. The trace tells you *how it got there*: which buckets the router opened in each lane, the scan fraction, and the exact ranked, dated facts handed to the reader.

In [ ]:
result = mem.answer("What did we change after the checkout latency incident?",
                    question_date="2024-12-01")
print("ANSWER:", result.answer)

t = result.trace
print(f"\nscanned {t.scan_fraction:.0%} of memory ({t.n_scanned}/{t.n_facts} facts), reader={t.reader_type}")
for lane, info in (t.lanes or {}).items():
    print(f"  {lane}: opened buckets {[b.get('bucket') if isinstance(b, dict) else b for b in info.get('buckets', [])]}"
          f", scan {info.get('scan_frac', 0):.0%}")

print("\nfacts the reader actually saw:")
for line in t.ranked_facts[:8]:
    print("  ", line[:110])

## Provenance: from a fact back to its source

`search()` returns evidence hits instead of an answer. Each hit carries `doc_name` + `source_ref`, and the local store can resolve that reference back to the exact passage in the original file.

In [ ]:
hits = mem.search("pgbouncer caveats", top_k=3).hits
for h in hits:
    print(f"{h.ref}  {h.fact[:90]}")
    print(f"        from: {h.doc_name} @ {h.source_ref}")

# Drill into the top hit's original passage
top = hits[0]
src = store.resolve_source(top.doc_id, top.source_ref, context=1)
print(f"\n--- source passage in {src['name']} (session {src['session']}):")
for i, turn in enumerate(src["turns"]):
    marker = ">>" if i == src.get("highlight") else "  "
    print(f"{marker} {turn['content'][:120]}")

## The scan budget: pay for what you read

The router opens buckets in order of routing probability until the budget (fraction of total facts) is spent. On a small corpus a full scan is cheap — the budget matters when memory grows to tens of thousands of facts, where scanning 30% instead of 100% cuts retrieval cost ~3x with minimal quality loss. Watch the trace change as we tighten it.

In [ ]:
question = "Which customers had problems related to API keys?"

for budget in (1.0, 0.5, 0.2):
    mem._retrieval.scan_budget = budget
    mem._retrieval.scan_budget_reason = budget
    r = mem.answer(question, question_date="2024-12-01")
    opened = sum(len(i.get("buckets", [])) for i in (r.trace.lanes or {}).values())
    print(f"budget {budget:.0%}: scanned {r.trace.scan_fraction:.0%}, "
          f"{opened} buckets opened -> {(r.answer or '')[:80]}")

## Where to go next

- `membukkit ui` — this same trace, rendered visually: memory map, ask panel, fact-to-source drill-down.
- `03_rag_mode.ipynb` — document QA over passage corpora with the same bucket-gated retrieval.
- `04_benchmarks.ipynb` — reproduce the benchmark numbers from the README.